# Grad-CAM Attention Heatmaps
Uses Grad-CAM to visualize which regions of an image the memorability model focuses on.
Also compares heatmaps side-by-side for original vs. blurred-face image pairs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/STATUN3106-image-memorability'):
    !git clone https://github.com/beccangruenberg/STATUN3106-image-memorability.git /content/STATUN3106-image-memorability
else:
    !git -C /content/STATUN3106-image-memorability pull


# ── DEPENDENCIES ────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from PIL import Image
from torchvision import transforms

# ── PATHS ────────────────────────────────────────────────────────────
DATA_PATH    = "/content/drive/MyDrive/STATUN3106 - Applied Machine Learning/Final Project/STATUN3106_Final_Project/Public/final_submission"
CHECKPOINT   = f"{DATA_PATH}/checkpoints/050626_215834_model_epoch3.pth"   # <-- update
ORIG_DIR     = f"{DATA_PATH}/data/lamem_final"         # <-- update if needed
BLURRED_DIR  = f"{DATA_PATH}/data/ablation_face_blur"          # <-- update if needed
SCORES_CSV  = f"{DATA_PATH}/results/blur/blur_scores.csv"       # output from blur notebook
RESULTS_DIR = f"{DATA_PATH}/results/heatmaps"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── MODEL ────────────────────────────────────────────────────────────
class MemorabilityModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('resnet50', pretrained=False, num_classes=0)
        self.regressor = nn.Sequential(
            nn.Linear(2048, 512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.regressor(self.backbone(x)).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = MemorabilityModel().to(device)
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()
print(f"Model loaded on: {device}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# ── GRAD-CAM ─────────────────────────────────────────────────────────
# Hooks into model.backbone.layer4[-1] — the last and most semantic ResNet block.
# Gradients flowing back from the memorability score tell us which spatial
# regions most influenced the prediction.

class GradCAM:
    def __init__(self, model):
        self.model       = model
        self._activations = None
        self._gradients   = None
        target = model.backbone.layer4[-1]
        target.register_forward_hook(self._save_act)
        target.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):
        self._activations = out.detach()

    def _save_grad(self, module, grad_in, grad_out):
        self._gradients = grad_out[0].detach()

    def generate(self, pil_img):
        """Returns (memorability_score float, cam array 224x224 normalized 0-1)."""
        self.model.eval()
        tensor = transform(pil_img).unsqueeze(0).to(device).requires_grad_(True)
        score  = self.model(tensor)
        self.model.zero_grad()
        score.backward()
        # Global average pool gradients → spatial weights
        weights = self._gradients.mean(dim=[2, 3], keepdim=True)
        # Weighted combination of feature maps + ReLU
        cam = F.relu((weights * self._activations).sum(dim=1, keepdim=True))
        # Upsample to input resolution
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return score.item(), cam

gradcam = GradCAM(model)

def make_overlay(pil_img, cam, alpha=0.45):
    """Blend a jet-colored CAM over the resized original image."""
    cmap    = matplotlib.colormaps['jet']
    cam_rgb = (cmap(cam)[:, :, :3] * 255).astype(np.uint8)
    base    = np.array(pil_img.resize((224, 224))).astype(np.float32)
    return ((1 - alpha) * base + alpha * cam_rgb).clip(0, 255).astype(np.uint8)

print("GradCAM ready.")

In [ ]:
# ── FIGURE 1: Single-image heatmap (original image) ──────────────────
# Pick any image to demo the full pipeline

filenames = sorted([
    f for f in os.listdir(ORIG_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

DEMO_FILE = filenames[0]   # change index or hardcode a filename
demo_img  = Image.open(os.path.join(ORIG_DIR, DEMO_FILE)).convert('RGB')
score, cam = gradcam.generate(demo_img)
overlay    = make_overlay(demo_img, cam)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(demo_img.resize((224, 224))); axes[0].set_title('Original'); axes[0].axis('off')
im = axes[1].imshow(cam, cmap='jet'); axes[1].set_title('Grad-CAM'); axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
fig.suptitle(f'{DEMO_FILE}  |  Score: {score:.4f}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig1_single_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── FIGURE 2: Original vs. Blurred heatmap comparison ────────────────
# For a single pair: shows whether the model's attention shifts
# when faces are removed.

def compare_heatmaps(filename, save=True):
    orig_img    = Image.open(os.path.join(ORIG_DIR, filename)).convert('RGB')
    blurred_img = Image.open(os.path.join(BLURRED_DIR, filename)).convert('RGB')

    s_orig, cam_orig = gradcam.generate(orig_img)
    s_blur, cam_blur = gradcam.generate(blurred_img)
    ov_orig = make_overlay(orig_img, cam_orig)
    ov_blur = make_overlay(blurred_img, cam_blur)

    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    for row_idx, (img, cam, overlay, label, sc) in enumerate([
        (orig_img,    cam_orig, ov_orig, 'Original', s_orig),
        (blurred_img, cam_blur, ov_blur, 'Blurred',  s_blur)
    ]):
        axes[row_idx, 0].imshow(img.resize((224, 224)))
        axes[row_idx, 0].set_title(f'{label} | Score: {sc:.3f}', fontsize=11, fontweight='bold')
        axes[row_idx, 0].axis('off')

        im = axes[row_idx, 1].imshow(cam, cmap='jet')
        axes[row_idx, 1].set_title('Grad-CAM', fontsize=11)
        axes[row_idx, 1].axis('off')
        plt.colorbar(im, ax=axes[row_idx, 1], fraction=0.046, pad=0.04)

        axes[row_idx, 2].imshow(overlay)
        axes[row_idx, 2].set_title('Overlay', fontsize=11)
        axes[row_idx, 2].axis('off')

    delta = s_blur - s_orig
    fig.suptitle(f'{filename}  |  Δ = {delta:+.4f}  ({"blur ↑ memorability" if delta > 0 else "blur ↓ memorability"})',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    if save:
        stem = filename.rsplit('.', 1)[0]
        plt.savefig(f'{RESULTS_DIR}/fig2_compare_{stem}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Run on the demo image first
compare_heatmaps(DEMO_FILE)

In [ ]:
# ── FIGURE 3: Grid of 6 heatmap overlays (varied memorability scores) ─
# Gives a broad visual survey of what the model attends to across the dataset.
# Picks 3 high-scoring and 3 low-scoring images for contrast.

# Score all originals first (or load from blur notebook CSV if available)
if os.path.exists(SCORES_CSV):
    df_scores = pd.read_csv(SCORES_CSV)
    print(f"Loaded {len(df_scores)} pre-computed scores from blur notebook.")
else:
    print("Scoring all originals — this may take a minute...")
    rows = []
    for f in filenames:
        img = Image.open(os.path.join(ORIG_DIR, f)).convert('RGB')
        with torch.no_grad():
            s = model(transform(img).unsqueeze(0).to(device)).item()
        rows.append({'filename': f, 'original': s})
    df_scores = pd.DataFrame(rows)

high3 = df_scores.nlargest(3, 'original')['filename'].tolist()
low3  = df_scores.nsmallest(3, 'original')['filename'].tolist()
survey = high3 + low3
labels = ['High'] * 3 + ['Low'] * 3

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for col, (fname, cat) in enumerate(zip(survey, labels)):
    img = Image.open(os.path.join(ORIG_DIR, fname)).convert('RGB')
    sc, cam = gradcam.generate(img)
    overlay = make_overlay(img, cam)

    axes[0, col].imshow(img.resize((224, 224)))
    axes[0, col].set_title(f'{cat}\nScore: {sc:.3f}', fontsize=9,
                            color='darkgreen' if cat == 'High' else 'darkred', fontweight='bold')
    axes[0, col].axis('off')

    axes[1, col].imshow(overlay)
    axes[1, col].set_title('Overlay', fontsize=9)
    axes[1, col].axis('off')

fig.suptitle('Grad-CAM: High Memorability (left) vs. Low Memorability (right)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig3_high_vs_low_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── FIGURE 4: Batch heatmap comparisons for top-affected pairs ────────
# Uses the blur_scores.csv from the face blur notebook to pick the
# pairs where score changed most — then shows heatmaps for all of them.

if os.path.exists(SCORES_CSV):
    df_blur = pd.read_csv(SCORES_CSV)
    # Top 3 biggest drops + top 3 biggest gains
    showcase = pd.concat([
        df_blur.nsmallest(3, 'delta'),
        df_blur.nlargest(3, 'delta')
    ])
    print(f"Running heatmap comparisons for {len(showcase)} pairs...")
    for _, row in showcase.iterrows():
        compare_heatmaps(row['filename'], save=True)
else:
    print("blur_scores.csv not found — run the face blur notebook first, or pick files manually below.")
    # Manual fallback: pick files directly
    manual_files = filenames[:3]   # <-- replace with specific filenames if desired
    for f in manual_files:
        compare_heatmaps(f, save=True)